# Land Registry Price Data (Oxfordshire 2015-2025) Structural Audit

#### Load data

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path

column_names = [
    "transaction_id",
    "price",
    "date_of_transfer",
    "postcode",
    "property_type",
    "old_new",
    "duration",
    "paon",
    "saon",
    "street",
    "locality",
    "town_city",
    "district",
    "county",
    "ppd_category_type",
    "record_status"
]
# df1 = pd.read_csv("../data/raw/pp-1995.csv", names=column_names)

folder = Path('../data/raw')
file_gen = folder.glob('ppd_data*.csv')
df = pd.concat([pd.read_csv(f, names=column_names) for f in file_gen])

In [ ]:
df['district'].value_counts()
df.head()

#### Investigating anomalies

Checking datatypes

In [ ]:
df.dtypes

Looking at data consistency with types

In [ ]:
df.head(5)

Checking null and unique values

In [ ]:

print(df.nunique())
print(df.isna().sum())
print(df.duplicated().sum())

Looking at min/max price

In [ ]:
print(df['price'].sort_values())
print((df['price'] < 1000).sum())
print((df['price'] > 100000000).sum())

Inspect values of street, town_city, locality, district, county

Searching for unexpected characters

In [ ]:
to_check = ['street', 'town_city', 'locality', 'district', 'county']
for n in to_check:
    print(f"Checking {n} : contains {df[n].str.count(r"[^&a-zA-Z0-9'.\s-]").sum()} special or accented characters excl -'&.")


Testing regex search

In [ ]:
import re
test = "!£$%^&*()<>?/@|¬"

re.findall(r"[^&a-zA-Z0-9'.\s-]", test)

Looking at actual flagged values

In [ ]:
for n in to_check:
    print(df[n][df[n].str.contains(r"[^&a-zA-Z0-9'.\s-]")].value_counts())


A list of the unique special characters being found

In [ ]:

results = set()
for n in to_check:
    flagged_rows = df[n][df[n].str.contains(r"[^&a-zA-Z0-9'.\s-]")]
    for r in flagged_rows:
       matches = re.findall(r"[^&a-zA-Z0-9'.\s-]", r)
       results.update(matches)
print(results)



Finding which day of the year had no sales

In [ ]:
# print(df['date_of_transfer'].sort_values())
df['date_of_transfer'] = df['date_of_transfer'].astype('datetime64[s]')
dates = pd.DataFrame()
dates['day'] = df['date_of_transfer'].dt.day
dates['month'] = df['date_of_transfer'].dt.month
print(dates.groupby('month')['day'].nunique())
print(dates[dates['month'] == 12].groupby('day').count())

Postcode check on structure

In [ ]:
print(f"Postcodes between {df['postcode'].str.len().max()} and {df['postcode'].str.len().min()} characters in length")


passed_postcodes = df['postcode'].str.match(r"^[A-Z]{1,2}\d[A-Z0-9]? \d[A-Z]{2}$", na=False).sum()
total_postcodes = df['postcode'].dropna().count()
print(f"{total_postcodes - passed_postcodes} postcodes failed the regex check")
print(df['postcode'][~df['postcode'].str.match(r"^[A-Z]{1,2}\d[A-Z0-9]? \d[A-Z]{2}$", na=True)])